# 05 · Trocar motores, configurar por JSON, reindexar

Cada peça é trocada só na criação do objeto; o resto do código não muda.

| Peça | Opções incluídas |
|---|---|
| Embedding | FastEmbed (padrão), Ollama, formato OpenAI, Gemini |
| Modelo de linguagem | Ollama (padrão), formato OpenAI, Gemini, Anthropic, qualquer função `str -> str` |
| Banco vetorial | Qdrant (padrão), Chroma, em memória |

> **Instalação** (uma vez):
> ```bash
> pip install "pi-memory[local]"
> ollama pull qwen2.5:14b
> ```
> Todos os dados deste notebook são fictícios. As saídas abaixo vêm de uma execução real.

In [1]:
import json
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # aviso do tqdm no Jupyter

def mostrar(obj, limite=1500):
    """JSON legível (o mesmo formato da biblioteca, do HTTP e do MCP)."""
    texto = json.dumps(obj, ensure_ascii=False, indent=2)
    print(texto if len(texto) <= limite else texto[:limite] + "\n…")

def pasta_limpa(nome):
    """Cada notebook começa do zero: apaga a pasta de dados da execução anterior."""
    shutil.rmtree(nome, ignore_errors=True)
    return nome

In [2]:
import os
from pi_memory import PiMemory
from pi_memory.embedders import OllamaEmbedder, OpenAIEmbedder
from pi_memory.stores import ChromaStore, MemoryStore

def demonstrar(pm, rotulo):
    pm.memories.remember("O usuário tem uma gata chamada Mel", owner="ana")
    pm.memories.remember("O usuário trabalha como enfermeira", owner="ana")
    melhor = pm.memories.search("qual o nome do animal de estimação?", owner="ana")["results"][0]
    print(f"{rotulo:<34} {pm.embedder.name:<42} → {melhor['text']}")
    pm.close()

## Embeddings e bancos diferentes, mesmo código

In [3]:
demonstrar(PiMemory(path=pasta_limpa("dados_05_qdrant"), llm=lambda p: ""), "FastEmbed + Qdrant (padrão)")
demonstrar(PiMemory(embedder=OllamaEmbedder("nomic-embed-text"),
                    store=ChromaStore(path=pasta_limpa("dados_05_chroma")), llm=lambda p: ""),
           "Ollama + Chroma")
demonstrar(PiMemory(embedder=OpenAIEmbedder("nomic-embed-text", base_url="http://localhost:11434/v1"),
                    store=MemoryStore(), llm=lambda p: ""),
           "formato OpenAI (via Ollama) + memória")

FastEmbed + Qdrant (padrão)        fastembed:intfloat/multilingual-e5-large   → O usuário tem uma gata chamada Mel


Ollama + Chroma                    ollama:nomic-embed-text                    → O usuário tem uma gata chamada Mel


formato OpenAI (via Ollama) + memória openai:http://localhost:11434/v1:nomic-embed-text → O usuário tem uma gata chamada Mel


## Gemini (faixa gratuita), se houver chave

A chave vem da variável de ambiente `GEMINI_API_KEY` e nunca aparece em mensagens. Os nomes
dos modelos de conversa mudam rápido, por isso `GeminiLLM` exige o nome.

In [4]:
if os.environ.get("GEMINI_API_KEY"):
    from pi_memory.embedders import GeminiEmbedder
    from pi_memory.llms import GeminiLLM
    demonstrar(PiMemory(embedder=GeminiEmbedder(), store=MemoryStore(), llm=lambda p: ""),
               "Gemini embedding + memória")
    pm = PiMemory(embedder=OllamaEmbedder("nomic-embed-text"), store=MemoryStore(),
                  llm=GeminiLLM("gemini-3.5-flash", timeout=120))
    mostrar(pm.memories.add("Sou o Bruno e moro em Vila Exemplo.", owner="bruno"))
    pm.close()
else:
    print("GEMINI_API_KEY não definida: parte do Gemini pulada")

Gemini embedding + memória         gemini:gemini-embedding-001                → O usuário tem uma gata chamada Mel


{
  "decisions": [
    {
      "event": "add",
      "memory_id": "3efe750d-6cd1-46f9-95a6-b012ab2d1a3c",
      "text": "O nome do usuário é Bruno",
      "previous_text": null,
      "topic": "perfil",
      "reason": null
    },
    {
      "event": "add",
      "memory_id": "f05419ff-a584-4927-a8d0-1be514f1b1cb",
      "text": "O usuário mora em Vila Exemplo",
      "previous_text": null,
      "topic": "casa",
      "reason": null
    }
  ]
}


## Qualquer função como modelo de linguagem

Útil para usar o SDK que você já tem, um modelo remoto próprio, ou para testes.

In [5]:
def meu_llm(prompt: str) -> str:
    # aqui entraria a chamada ao seu provedor; esta resposta fixa só mostra o formato
    return "FATO: O usuário gosta de jazz" if "Abaixo está um trecho" in prompt else "1. lazer"

pm = PiMemory(embedder=OllamaEmbedder("nomic-embed-text"), store=MemoryStore(), llm=meu_llm)
mostrar(pm.memories.add("adoro jazz", owner="ana"))
pm.close()

{
  "decisions": [
    {
      "event": "add",
      "memory_id": "813e8109-0583-4c4c-a35c-671daca722e0",
      "text": "O usuário gosta de jazz",
      "previous_text": null,
      "topic": "lazer",
      "reason": null
    }
  ]
}


## Configuração em JSON

O mesmo dicionário serve para a linha de comando (`--config arquivo.json`) e para os servidores.
Chaves de API **não** entram no arquivo.

In [6]:
config = {
    "path": pasta_limpa("dados_05_config"),
    "embedder": {"type": "ollama", "model": "nomic-embed-text"},
    "llm": {"type": "ollama", "model": "qwen2.5:14b", "timeout": 180},
    "store": {"type": "chroma", "path": "dados_05_config/chroma"},
    "topics": ["trabalho", "família", "saúde", "lazer", "outros"],
}
pm = PiMemory.from_config(config)
print(type(pm.embedder).__name__, type(pm.llm).__name__, type(pm.store).__name__, pm.topics)
pm.close()

from pi_memory import PiMemoryError
try:
    PiMemory.from_config({"llm": {"type": "openai", "model": "x", "api_key": "sk-..."}})
except PiMemoryError as e:
    print(e.code, "→", e.message)

OllamaEmbedder OllamaLLM ChromaStore ('trabalho', 'família', 'saúde', 'lazer', 'outros')
invalid_input → Chaves de API não vão no arquivo de configuração (api_key): defina a variável de ambiente do motor (OPENAI_API_KEY, GEMINI_API_KEY, ANTHROPIC_API_KEY).


## Trocar de embedding numa memória existente

Vetores de modelos diferentes não se misturam: abrir com outro modelo é recusado com
explicação. Para migrar, abra com o modelo antigo e chame `reindex`.

In [7]:
pasta = pasta_limpa("dados_05_reindex")
pm = PiMemory(path=pasta, llm=lambda p: "")
pm.memories.remember("O usuário tem uma gata chamada Mel", owner="ana")
pm.close()

try:
    PiMemory(path=pasta, embedder=OllamaEmbedder("nomic-embed-text"), llm=lambda p: "")
except PiMemoryError as e:
    print(e.code, "→", e.message, "\n")

pm = PiMemory(path=pasta, llm=lambda p: "")
mostrar(pm.reindex(OllamaEmbedder("nomic-embed-text")))
pm.close()

pm = PiMemory(path=pasta, embedder=OllamaEmbedder("nomic-embed-text"), llm=lambda p: "")
print(pm.memories.search("gato", owner="ana")["results"][0]["text"])
pm.close()

embedder_mismatch → A coleção 'pi_memory_memories' foi criada com o embedding 'fastembed:intfloat/multilingual-e5-large' (1024 dimensões), mas agora o embedding é 'ollama:nomic-embed-text' (768 dimensões). Vetores de modelos diferentes não se misturam: volte ao modelo anterior, ou abra com ele e chame pm.reindex(novo_embedder) para regravar tudo com o novo. 



{
  "memories": 1,
  "chunks": 0,
  "embedder": "ollama:nomic-embed-text"
}


O usuário tem uma gata chamada Mel
